# MCP con LangChain y Ollama

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion6/2-mcp-con-langchain-y-ollama.ipynb)

En este segundo notebook reutilizaremos los mismos dos casos del notebook anterior, pero ahora exponiendo las capacidades vía MCP. La meta es que el estudiante vea el cambio arquitectónico con el mismo problema de negocio: primero conectaremos un cliente MCP explícito para entender el protocolo y luego cargaremos esas herramientas dentro de un agente que las consume a través de LangChain.

### Referencias
- [MCP Python SDK](https://py.sdk.modelcontextprotocol.io/)
- [LangChain MCP Adapters](https://pypi.org/project/langchain-mcp-adapters/)
- [Open-Meteo API](https://open-meteo.com/)
- [Ollama](https://ollama.com/)


In [ ]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages


In [ ]:
!test '{IN_COLAB}' = 'True' && pip install mcp langchain langchain-core langchain-ollama langchain-mcp-adapters langgraph httpx ollama colab-xterm


### Cargando a Ollama

Usaremos el mismo modelo local de la lección anterior para que el contraste se concentre en la arquitectura y no en cambiar de modelo.


In [ ]:
!sudo apt install zstd -y
!if ! type ollama > /dev/null; then curl -fsSL https://ollama.com/install.sh | sh; else echo "Ollama ya está instalado."; fi


## Atención

En Colab, si el servidor de Ollama no está levantado todavía, inícialo en la terminal embebida. Si corres en local y ya tienes `ollama serve`, basta con continuar.

In [ ]:
%load_ext colabxterm
%xterm


Mantendremos `llama3.2:3b` para la demostración.


In [ ]:
!ollama pull llama3.2:3b


## Paso 1: escribimos dos servidores MCP pequeños

En lugar de registrar las herramientas dentro del notebook principal, ahora las publicaremos como servidores MCP sobre `stdio`. Eso permite que otra aplicación las descubra, lea su esquema y las use sin acoplarse directamente a nuestras funciones de Python.

In [ ]:
from pathlib import Path

SERVERS_DIR = Path.cwd() / 'mcp_servers'
SERVERS_DIR.mkdir(exist_ok=True)
SERVERS_DIR


In [ ]:
calculator_server = SERVERS_DIR / 'calculator_mcp_server.py'
calculator_server.write_text('from mcp.server.fastmcp import FastMCP\nimport ast\nimport operator as op\n\nmcp = FastMCP("CalculadoraMCP")\n\nALLOWED_BIN_OPS = {\n    ast.Add: op.add,\n    ast.Sub: op.sub,\n    ast.Mult: op.mul,\n    ast.Div: op.truediv,\n    ast.Pow: op.pow,\n    ast.Mod: op.mod,\n}\nALLOWED_UNARY_OPS = {\n    ast.UAdd: op.pos,\n    ast.USub: op.neg,\n}\n\ndef evaluate_expression(expression: str):\n    def _eval(node):\n        if isinstance(node, ast.Expression):\n            return _eval(node.body)\n        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):\n            return node.value\n        if isinstance(node, ast.BinOp) and type(node.op) in ALLOWED_BIN_OPS:\n            return ALLOWED_BIN_OPS[type(node.op)](_eval(node.left), _eval(node.right))\n        if isinstance(node, ast.UnaryOp) and type(node.op) in ALLOWED_UNARY_OPS:\n            return ALLOWED_UNARY_OPS[type(node.op)](_eval(node.operand))\n        raise ValueError("La expresión contiene operadores no soportados.")\n\n    return _eval(ast.parse(expression, mode="eval"))\n\n@mcp.tool()\ndef calculadora(expression: str) -> str:\n    """Evalúa una expresión aritmética segura con +, -, *, /, %, ** y paréntesis."""\n    result = evaluate_expression(expression)\n    return f"Resultado exacto: {result}"\n\nif __name__ == "__main__":\n    mcp.run(transport="stdio")\n', encoding='utf-8')
print(calculator_server.read_text(encoding='utf-8'))


In [ ]:
weather_server = SERVERS_DIR / 'weather_mcp_server.py'
weather_server.write_text('from mcp.server.fastmcp import FastMCP\nimport httpx\n\nmcp = FastMCP("ClimaMCP")\n\n@mcp.tool()\ndef clima_actual(city: str) -> str:\n    """Consulta el clima actual de una ciudad usando Open-Meteo."""\n    geocode_response = httpx.get(\n        "https://geocoding-api.open-meteo.com/v1/search",\n        params={"name": city, "count": 1, "language": "es", "format": "json"},\n        timeout=30.0,\n    )\n    geocode_response.raise_for_status()\n    geocode_data = geocode_response.json()\n    if not geocode_data.get("results"):\n        return f"No encontré información para la ciudad: {city}"\n\n    location = geocode_data["results"][0]\n    weather_response = httpx.get(\n        "https://api.open-meteo.com/v1/forecast",\n        params={\n            "latitude": location["latitude"],\n            "longitude": location["longitude"],\n            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",\n            "timezone": "auto",\n        },\n        timeout=30.0,\n    )\n    weather_response.raise_for_status()\n    weather_data = weather_response.json()["current"]\n\n    return (\n        f"Clima actual en {location[\'name\']}, {location.get(\'country\', \'\')}: "\n        f"temperatura {weather_data[\'temperature_2m\']}°C, "\n        f"humedad {weather_data[\'relative_humidity_2m\']}%, "\n        f"viento {weather_data[\'wind_speed_10m\']} km/h, "\n        f"weather_code {weather_data[\'weather_code\']}."\n    )\n\nif __name__ == "__main__":\n    mcp.run(transport="stdio")\n', encoding='utf-8')
print(weather_server.read_text(encoding='utf-8'))


## Paso 2: usamos un cliente MCP explícito

Antes de conectarlo a un agente, vale la pena mirar el protocolo casi sin ayudas. Inicializamos una sesión, listamos herramientas y ejecutamos una de ellas. Así se vuelve más claro qué parte pertenece al servidor y qué parte pertenece al host o cliente.

In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def probar_calculadora_mcp():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(calculator_server)],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            result = await session.call_tool('calculadora', {'expression': '(125 * 17) + 938'})
            return tools, result

tools_info, calc_result = await probar_calculadora_mcp()
tools_info, calc_result


In [ ]:
if hasattr(calc_result, 'structured_content'):
    calc_result.structured_content
else:
    calc_result.content


Esa llamada ya es MCP real: hay un servidor, un transporte, un cliente y una invocación de tool definida por un contrato común. Todavía no hay agente, pero ya existe interoperabilidad.

## Paso 3: capa híbrida con un agente consumidor de herramientas MCP

Ahora sí reintroducimos la experiencia de agente. La diferencia es que las herramientas ya no están embebidas en este notebook: vienen publicadas por servidores MCP y el agente las descubre a través del adaptador de LangChain.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

MODEL = 'llama3.2:3b'
llm = ChatOllama(model=MODEL, temperature=0)

client = MultiServerMCPClient(
    {
        'calculadora': {
            'transport': 'stdio',
            'command': sys.executable,
            'args': [str(calculator_server)],
        },
        'clima': {
            'transport': 'stdio',
            'command': sys.executable,
            'args': [str(weather_server)],
        },
    }
)

mcp_tools = await client.get_tools()
agent_mcp = create_react_agent(model=llm, tools=mcp_tools)

async def preguntar_via_mcp(question: str):
    result = await agent_mcp.ainvoke({'messages': [('user', question)]})
    final_message = result['messages'][-1]
    return final_message.content, result


## Caso 1: la calculadora ahora viaja por MCP


In [ ]:
respuesta_calculo_mcp, traza_calculo_mcp = await preguntar_via_mcp(
    '¿Cuánto es (125 * 17) + 938? Responde en español y menciona el resultado final.'
)
print(respuesta_calculo_mcp)


## Caso 2: clima actual consumido como tool MCP


In [ ]:
respuesta_clima_mcp, traza_clima_mcp = await preguntar_via_mcp(
    'Consulta el clima actual de Cali, Colombia, y resume la información más importante en una oración.'
)
print(respuesta_clima_mcp)


In [ ]:
[tool.name for tool in mcp_tools]


## Conclusiones

- En el notebook anterior ya teníamos herramientas útiles; aquí esas capacidades se transformaron en servicios interoperables.
- MCP separa mejor responsabilidades: el servidor expone capacidades y el host decide cómo consumirlas.
- La capa híbrida con LangChain ayuda a enseñar el flujo sin esconder el protocolo, mientras que la sección con `ClientSession` deja visible la mecánica básica de MCP.
